# CKD Edge Evaluation — MacBook M2 Max

End-to-end edge deployment evaluation on MacBook M2 Max (Apple Silicon).

**Pipeline:**
1. Cell 1: Environment + GPU/ANE check
2. Cell 2: Install dependencies (uv preferred, falls back to pip)
3. Cell 3: Verify mirror layout (run `setup_macbook_mirror.py` first!)
4. Cell 4: Pilot conversion + numerical sanity check (1 checkpoint)
5. Cell 5: Mass conversion (9 checkpoints x 4 formats)
6. Cell 6: PyTorch CPU baseline AUC (reference)
7. Cell 7: AUC evaluation — TFLite (fp32 + int8) on 3 test splits
8. Cell 8: AUC evaluation — CoreML (fp32 + int8) on ANE
9. Cell 9: Latency benchmarks (TFLite + CoreML, multiple compute units)
10. Cell 10: Aggregate results into thesis-ready Markdown + JSON

**Prerequisite:** Run `python scripts/edge/setup_macbook_mirror.py ...` FIRST to extract DF40 ZIPs locally + stage checkpoints + rewrite CSV paths. See `docs/EDGE_EVAL_MACBOOK_GUIDE.md` for the exact commands.

**Caveat:** M2 Max is a development-class chip, NOT a representative edge device. Latency on M2 Max will be 4-8x faster than on a Snapdragon 7-series CPU. Report numbers with this caveat in the thesis.


## 1. Environment + Hardware Check

In [ ]:
import platform
import sys

print('Python   :', sys.version.split()[0])
print('Platform :', platform.platform())
print('Machine  :', platform.machine())

# Confirm Apple Silicon
assert platform.machine() == 'arm64', (
    'This notebook expects Apple Silicon (arm64). '
        f'Got {platform.machine()!r}. CoreML benchmarks will not run.'
)
assert platform.system() == 'Darwin', (
    f'macOS expected, got {platform.system()!r}.'
)

# Show some system stats
import os
print('CPU count:', os.cpu_count())
try:
    import psutil
    mem = psutil.virtual_memory()
    print(f'RAM      : {mem.total / 1e9:.1f} GB total, {mem.available / 1e9:.1f} GB available')
except ImportError:
    pass


## 2. Install Dependencies

Use `uv` if available (fast), otherwise plain pip. **This cell can be skipped if you've already installed `requirements-macbook.txt`.**


In [ ]:
import shutil
import subprocess
import sys

REPO_ROOT = '.'  # adjust if running from elsewhere
REQ_FILE = f'{REPO_ROOT}/requirements-macbook.txt'

pkg_mgr = 'uv' if shutil.which('uv') else 'pip'
print(f'Using package manager: {pkg_mgr}')

if pkg_mgr == 'uv':
    subprocess.run(['uv', 'pip', 'install', '-r', REQ_FILE], check=True)
    subprocess.run(['uv', 'pip', 'install', '-e', REPO_ROOT], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', REQ_FILE], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_ROOT], check=True)

# Verify critical imports
import torch, onnx, onnx2tf  # noqa: F401
try:
    import coremltools
    print(f'coremltools version: {coremltools.__version__}')
except ImportError:
    print('coremltools NOT installed — CoreML cells will be skipped')
print(f'torch version       : {torch.__version__}')


## 3. Verify Local Mirror

Confirms `setup_macbook_mirror.py` was run and all expected paths exist.
If any check fails, run the setup script before proceeding.


In [ ]:
from pathlib import Path

# Adjust if you used a different --output-root
OUTPUT_ROOT = Path.home() / 'ckd-edge'
MIRROR_ROOT = OUTPUT_ROOT / 'mirror'
REPO_ROOT = Path('.').resolve()
CONFIG_PATH = REPO_ROOT / 'configs' / 'macbook.yaml'

checks = {
    'Output root': OUTPUT_ROOT.is_dir(),
    'Mirror root': MIRROR_ROOT.is_dir(),
    'configs/macbook.yaml': CONFIG_PATH.is_file(),
    'mirror/datasets/splits/gen3_test.csv': (MIRROR_ROOT / 'datasets/splits/gen3_test.csv').is_file(),
    'mirror/datasets/splits/gen1_test.csv': (MIRROR_ROOT / 'datasets/splits/gen1_test.csv').is_file(),
    'mirror/datasets/splits/gen2_test.csv': (MIRROR_ROOT / 'datasets/splits/gen2_test.csv').is_file(),
    'mirror/checkpoints/students/gen3_replay+ewc_seed0/best.pth': (
        MIRROR_ROOT / 'checkpoints/students/gen3_replay+ewc_seed0/best.pth'
    ).is_file(),
    'gen3 frames (df40/sd2.1/)': (OUTPUT_ROOT / 'df40/sd2.1').is_dir(),
    'df40_real/ff_real/': (OUTPUT_ROOT / 'df40_real/ff_real').is_dir(),
}
for k, ok in checks.items():
    print(('OK  ' if ok else 'FAIL'), k)
if not all(checks.values()):
    raise SystemExit('Mirror verification failed — run scripts/edge/setup_macbook_mirror.py first.')

# Make repo modules importable
if str(REPO_ROOT) not in __import__('sys').path:
    __import__('sys').path.insert(0, str(REPO_ROOT))


## 4. Pilot Conversion + Sanity Check

Convert ONE checkpoint to all 4 formats (TFLite fp32, TFLite int8, CoreML fp32, CoreML int8) and verify numerical match against the PyTorch reference. **Stop here if any sanity check fails** — debug before mass conversion.

Expected thresholds:
- TFLite fp32: max_abs_diff < 5e-4
- TFLite int8: max_abs_diff < 2e-1
- CoreML fp32: max_abs_diff < 5e-4
- CoreML int8: max_abs_diff < 2e-1


In [ ]:
from pathlib import Path

from src.evaluation.edge_eval import convert_to_tflite, export_onnx
from src.evaluation.edge_eval_sanity import verify_coreml, verify_tflite, SanityReport
from src.models.students.mobilenetv3 import build_student
from src.utils.checkpoint import load_checkpoint
from src.utils.config import load_config
from src.data.dataloader import build_dataloader
import pandas as pd
import numpy as np

cfg = load_config(str(CONFIG_PATH))
drive_root = Path(cfg['paths']['drive_root']).expanduser().resolve()
student_cfg = cfg['student']
aug_cfg = cfg['data']['augmentation']
image_size = int(student_cfg.get('input_size', 224))

# Pilot = gen3 final seed-0 (the headline checkpoint).
PILOT_TAG = 'gen3_replay+ewc_seed0'
pilot_ckpt = drive_root / 'checkpoints' / 'students' / PILOT_TAG / 'best.pth'
print('Pilot checkpoint:', pilot_ckpt)

model = build_student(
    hidden_dim=int(student_cfg.get('head', {}).get('hidden_dim', 256)),
    dropout=float(student_cfg.get('head', {}).get('dropout', 0.2)),
    num_classes=int(student_cfg.get('num_classes', 2)),
    pretrained=False,
)
meta = load_checkpoint(pilot_ckpt, model=model, strict=False, map_location='cpu')
print(f'Loaded epoch={meta.get("epoch")} best_val_auc={meta.get("best_val_auc")}')
model.eval()

# Build small mixed calibration loader.
rng = np.random.default_rng(0)
chunks = []
for gen in ('gen1', 'gen2', 'gen3'):
    src = drive_root / 'datasets' / 'splits' / f'{gen}_test.csv'
    df = pd.read_csv(src, low_memory=False)
    chunks.append(df.sample(n=min(200, len(df)), random_state=0))
calib_csv = OUTPUT_ROOT / 'edge_results' / 'subsets' / 'calibration_pilot.csv'
calib_csv.parent.mkdir(parents=True, exist_ok=True)
pd.concat(chunks, axis=0).reset_index(drop=True).to_csv(calib_csv, index=False)
calib_loader = build_dataloader(
    csv_path=calib_csv, mode='test', batch_size=32,
    soft_label_path=None, image_size=image_size, aug_cfg=aug_cfg,
    num_workers=2, pin_memory=False,
)
print(f'Calibration loader: {len(calib_csv.read_text().splitlines()) - 1} rows')

# Conversion outputs
CONVERSION_ROOT = OUTPUT_ROOT / 'edge_results' / 'conversions' / PILOT_TAG
CONVERSION_ROOT.mkdir(parents=True, exist_ok=True)

# Export ONNX
onnx_path = export_onnx(model, CONVERSION_ROOT / 'model.onnx', input_size=image_size, opset=17)
print('ONNX exported:', onnx_path)

# Convert to TFLite (fp32 + int8)
tflite_paths = convert_to_tflite(
    onnx_path,
    CONVERSION_ROOT / 'tflite',
    modes=('fp32', 'int8'),
    representative_loader=calib_loader,
    calibration_batches=10,
)
print('TFLite outputs:')
for mode, p in tflite_paths.items():
    print(f'  {mode}: {p} ({p.stat().st_size / 1e6:.2f} MB)')

# Convert to CoreML (fp32 + int8) — only on macOS
coreml_paths = {}
if platform.system() == 'Darwin':
    from src.evaluation.edge_eval_coreml import export_coreml
    for mode in ('fp32', 'int8'):
        try:
            p = export_coreml(
                model,
                CONVERSION_ROOT / 'coreml' / f'{mode}.mlpackage',
                input_size=image_size,
                mode=mode,
                compute_precision='FLOAT32' if mode == 'fp32' else 'FLOAT16',
            )
            coreml_paths[mode] = p
        except Exception as exc:
            print(f'CoreML {mode} conversion failed: {exc}')
    print('CoreML outputs:')
    for mode, p in coreml_paths.items():
        print(f'  {mode}: {p}')

# --- Sanity check ---
sample = np.random.default_rng(42).standard_normal((4, 3, image_size, image_size)).astype(np.float32)
tf_report = verify_tflite(model, tflite_paths, sample_inputs=sample)
print('TFLite sanity:')
print(tf_report.summary())
if coreml_paths:
    cm_report = verify_coreml(model, coreml_paths, sample_inputs=sample)
    print('CoreML sanity:')
    print(cm_report.summary())
    if not (tf_report.all_passed() and cm_report.all_passed()):
        print('\nWARNING: Some sanity checks failed. Inspect drift before trusting downstream results.')
elif not tf_report.all_passed():
    print('\nWARNING: TFLite sanity failed.')


## 5. Mass Conversion (9 checkpoints x up to 4 formats)

Runs `scripts/edge/run_edge_eval_macbook.py` end-to-end. Long cell (~30-60 min depending on chip). Output: converted artifacts + full AUC + latency JSON + Markdown summary.

To debug individual steps, prefer the `--pilot-only` flag from CLI; the full run is here for one-click execution.


In [ ]:
import subprocess, sys

result = subprocess.run(
    [
        sys.executable,
        'scripts/edge/run_edge_eval_macbook.py',
        '--config', str(CONFIG_PATH),
        '--output-dir', str(OUTPUT_ROOT / 'edge_results'),
        '--subset-rows', '5000',
        '--num-latency-runs', '200',
        '--num-warmup', '20',
        '--batch-size', '32',
        '--num-workers', '2',
    ],
    cwd=str(REPO_ROOT),
)
if result.returncode != 0:
    print(f'Run failed with exit code {result.returncode}. Inspect the log above.')
else:
    print('Edge evaluation completed.')


## 6. Inspect Aggregated Results

Read the JSON output and print the key tables for thesis use.


In [ ]:
import json
from pathlib import Path
import pandas as pd

RESULTS_JSON = OUTPUT_ROOT / 'edge_results' / 'results' / 'edge_eval_full.json'
RESULTS_MD   = OUTPUT_ROOT / 'edge_results' / 'results' / 'edge_eval_summary.md'

if not RESULTS_JSON.is_file():
    raise FileNotFoundError(f'Results not found at {RESULTS_JSON} — Cell 5 must complete first.')

data = json.loads(RESULTS_JSON.read_text())
print('Run started :', data['started_at'])
print('Run finished:', data['finished_at'])
print(f'Elapsed     : {data["elapsed_seconds"] / 60:.1f} min')
print(f'CoreML      : {"enabled" if data["coreml_enabled"] else "disabled"}')
print(f'Checkpoints : {data["num_checkpoints_converted"]}')
print()

# Sanity table
san_df = pd.DataFrame(data['sanity_report'])
if not san_df.empty:
    print('=== Sanity check ===')
    print(san_df.to_string(index=False))
    print()

# Latency
lat_df = pd.DataFrame(data['latency'])
if not lat_df.empty:
    print('=== Latency (single-image, batch=1) ===')
    print(
        lat_df[['runtime', 'mode', 'compute_unit', 'latency_ms_mean',
                'latency_ms_p50', 'latency_ms_p95', 'latency_ms_p99',
                'size_mb']].to_string(index=False)
    )
    print()

# AUC aggregated by (ckpt_gen, runtime, mode, test_split)
ev_df = pd.DataFrame(data['evaluation'])
if not ev_df.empty:
    ev_df['ckpt_gen'] = ev_df['run_tag'].str.extract(r'^(gen\d)')[0]
    agg = (
        ev_df.groupby(['ckpt_gen', 'runtime', 'mode', 'test_split'])['auc']
        .agg(['mean', 'std', 'count'])
        .round(4)
        .reset_index()
    )
    print('=== AUC mean +/- std by group ===')
    print(agg.to_string(index=False))
    print()

print(f'Full Markdown summary: {RESULTS_MD}')


## 7. Optional: Quick Figure (Latency vs Size)

Generates a quick scatter plot of model size vs latency, color-coded by runtime/mode. Saves to `edge_results/results/latency_vs_size.png`.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

lat_df = pd.DataFrame(data['latency'])
if lat_df.empty:
    print('No latency data — skipping figure.')
else:
    fig, ax = plt.subplots(figsize=(7, 5))
    color_map = {
        ('tflite', 'fp32'): 'tab:blue',
        ('tflite', 'int8'): 'tab:cyan',
        ('coreml', 'fp32'): 'tab:orange',
        ('coreml', 'int8'): 'tab:red',
    }
    for _, row in lat_df.iterrows():
        key = (row['runtime'], row['mode'])
        ax.errorbar(
            row['size_mb'], row['latency_ms_mean'],
            yerr=[[row['latency_ms_mean'] - row['latency_ms_p50']],
                  [row['latency_ms_p95'] - row['latency_ms_mean']]],
            fmt='o', color=color_map.get(key, 'gray'),
            label=f"{row['runtime']}/{row['mode']}/{row['compute_unit']}",
            capsize=4,
        )
    ax.set_xlabel('Model size (MB)')
    ax.set_ylabel('Latency mean (ms)')
    ax.set_title('Edge Deployment: Latency vs Size (M2 Max)')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=8)
    fig.tight_layout()
    out_png = OUTPUT_ROOT / 'edge_results' / 'results' / 'latency_vs_size.png'
    fig.savefig(out_png, dpi=150)
    print(f'Figure saved: {out_png}')
    plt.show()


## 8. Open Final Summary

Open the Markdown summary file — copy/paste relevant tables into the thesis or use the figures generated above.


In [ ]:
print(f'Markdown summary path: {RESULTS_MD}')
print()
print('Preview:')
print('=' * 60)
print(RESULTS_MD.read_text()[:4000])
print('...')
